# LeRobot Dataset Replay for RBY1

LeRobot v2 형식 데이터셋(parquet + video)을 로드하여 실제 rby1 로봇에서 replay합니다.

## 주요 기능
- **Joint-space (16D)** 또는 **EEF-space (7D)** 데이터셋 자동 감지
- Joint 모드: `JointPositionCommand`로 직접 replay
- EEF 모드: FK→SE3→`CartesianCommand`로 replay (SDK 내부 IK)
- Joint 데이터셋을 EEF 모드로 on-the-fly 변환 가능
- 에피소드 선택, 속도 조절, 구간 지정 지원
- 실행 전 시각화 + 실행 후 tracking error 분석

## 데이터셋 형식
- **Joint (16D)**: `[right_arm(7), left_arm(7), right_gripper(1), left_gripper(1)]`
- **EEF (7D)**: `[x, y, z, roll, pitch, yaw, right_gripper]` (base frame, extrinsic XYZ Euler)

## Notebook 실행 순서
| Phase | Cell | 내용 |
|-------|------|------|
| **Phase 1** | 1~4 | 데이터셋 로드 및 시각화 (로봇 불필요) |
| **Phase 2** | 5~9 | 로봇 연결 → 초기자세 → 그리퍼 → Replay 실행 |
| **Phase 3** | 10 | Replay 후 분석 + Cleanup |

---
## Step 1 — 설정 (Configuration)

In [ ]:
from __future__ import annotations

import json
import logging
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)

# =====================================================================
# User Config
# =====================================================================

# 데이터셋 경로 (LeRobot v2 형식)
DATASET_DIR = Path("/home/hyunjin/rby1_ws/rby1_demo/LeRobotDataset_v2/PuttingCupintotheDishV2_50_eef")

# 에피소드 선택 (None이면 아래 브라우저에서 선택)
EPISODE_INDEX = 10

# Replay 모드: "joint" 또는 "eef"
#   - joint: JointPositionCommand로 replay (16D 데이터셋 필요)
#   - eef:   CartesianCommand로 replay (7D 데이터셋 또는 16D→FK 변환)
REPLAY_MODE = "eef"  # "joint" or "eef"

# ── 로봇 모드 ─────────────────────────────────────────────────────────
# "real" : 실제 rby1 로봇에 연결
# "sim"  : 시뮬레이터(Docker gRPC)에 연결 — 그리퍼/플랜지 전압 skip
ROBOT_MODE = "real"   # "real" or "sim"

# 로봇 연결 설정
ROBOT_IP    = "192.168.0.50:50051"   # 실로봇 IP
SIM_IP      = "localhost:50052"      # 시뮬레이터 IP (Docker)
ROBOT_MODEL = "a"

# 실제 연결 주소 (자동 결정)
_connect_addr = ROBOT_IP if ROBOT_MODE == "real" else SIM_IP

# Replay 파라미터
REPLAY_FPS = 15          # 원본 데이터 fps (meta/info.json에서 자동 로드)
REPLAY_SPEED = 1.0       # 재생 속도 배율 (0.5 = 절반 속도, 2.0 = 두배속)
FRAME_START = None       # 시작 프레임 (None = 처음부터)
FRAME_END = None         # 종료 프레임 (None = 끝까지)

# 로봇 제어 파라미터
COMMAND_PRIORITY = 10
HOLD_TIME = 0.2
MINIMUM_TIME = 0.05      # JointPositionCommand minimum_time

# EEF CartesianCommand 파라미터
EEF_TRANSLATION_WEIGHT = 1.0
EEF_ROTATION_WEIGHT = 1.0
EEF_VELOCITY_SCALING = 0.8

# Gripper 설정 (sim 모드에서는 자동 skip)
USE_REMOTE_GRIPPER = True

# rby1-data-collection repo를 sys.path에 추가 (remote_gripper 등 임포트용)
_data_collection_dir = str(Path("/home/hyunjin/rby1_ws/rby1-data-collection"))
if _data_collection_dir not in sys.path:
    sys.path.insert(0, _data_collection_dir)

# rby1 joint layout (24-DOF)
TORSO_SLICE = slice(2, 8)    # 6 joints
RIGHT_SLICE = slice(8, 15)   # 7 joints
LEFT_SLICE = slice(15, 22)   # 7 joints

# 데이터 수집 시 고정 torso 각도
TORSO_FIXED_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
TORSO_FIXED_RAD = np.deg2rad(TORSO_FIXED_DEG)

# Initial pose (degrees)
TORSO_INIT_DEG = np.array([0.0, 20.0, -40.0, 35.0, 0.0, 0.0], dtype=np.float64)
INIT_RIGHT_DEG = np.array([-24.0, -60.0, 10.0, -120.0, -60.0, 85.0, 0.0], dtype=np.float64)
INIT_LEFT_DEG  = np.array([-24.0,  60.0, -10.0, -120.0,  60.0, 85.0, 0.0], dtype=np.float64)

TORSO_INIT = np.deg2rad(TORSO_INIT_DEG)
INIT_RIGHT  = np.deg2rad(INIT_RIGHT_DEG)
INIT_LEFT   = np.deg2rad(INIT_LEFT_DEG)

# safe path midpoints
MID1_RIGHT = np.deg2rad([ 70.0, -30.0,   0.0, -100.0,  0.0, -70.0, 0.0])
MID1_LEFT  = np.deg2rad([ 70.0,  30.0,   0.0, -100.0,  0.0, -70.0, 0.0])
MID2_RIGHT = np.deg2rad([  0.0, -15.0,   0.0, -100.0,  0.0, -70.0, 0.0])
MID2_LEFT  = np.deg2rad([  0.0,  15.0,   0.0, -100.0,  0.0, -70.0, 0.0])

ELBOW_THRESHOLD_DEG = 90.0
INIT_DT = 0.05
INIT_HOLD_TIME = 0.2

print(f"Dataset:     {DATASET_DIR}")
print(f"Replay mode: {REPLAY_MODE}")
print(f"Robot mode:  [{ROBOT_MODE.upper()}]  →  {_connect_addr}  (model={ROBOT_MODEL})")
print(f"Speed:       {REPLAY_SPEED}x")


---
## Step 2 — 데이터셋 로드 및 메타데이터 확인

LeRobot v2.1 형식의 `meta/info.json`에서 state/action 차원을 읽어 자동으로 **Joint (16D)** vs **EEF (7D)** 데이터셋을 감지합니다.

In [ ]:
# ---------- 데이터셋 메타데이터 로드 ----------
assert DATASET_DIR.exists(), f"Dataset not found: {DATASET_DIR}"

meta_dir = DATASET_DIR / "meta"
assert meta_dir.exists(), f"meta/ not found: {meta_dir}"

# info.json
with open(meta_dir / "info.json") as f:
    info = json.load(f)

total_episodes = info["total_episodes"]
total_frames = info["total_frames"]
dataset_fps = info["fps"]
REPLAY_FPS = dataset_fps  # override with actual fps

# state/action 차원 자동 감지
features = info["features"]
# state column: "state" or "observation.state"
if "state" in features:
    STATE_COL = "state"
elif "observation.state" in features:
    STATE_COL = "observation.state"
else:
    raise KeyError(f"state column not found in features: {list(features.keys())}")

# action column: "action" or "actions"
if "action" in features:
    ACTION_COL = "action"
elif "actions" in features:
    ACTION_COL = "actions"
else:
    raise KeyError(f"action column not found in features: {list(features.keys())}")

state_shape = features[STATE_COL]["shape"]
action_shape = features[ACTION_COL]["shape"]
state_dim = state_shape[0]
action_dim = action_shape[0]

# 데이터셋 타입 자동 감지
if state_dim == 16:
    DATASET_TYPE = "joint"
elif state_dim == 7:
    DATASET_TYPE = "eef"
else:
    raise ValueError(f"Unknown state dimension: {state_dim} (expected 16 for joint or 7 for EEF)")

print(f"{'='*60}")
print(f"Dataset: {DATASET_DIR.name}")
print(f"Type: {DATASET_TYPE.upper()} (state_dim={state_dim}, action_dim={action_dim})")
print(f"Episodes: {total_episodes}  |  Total frames: {total_frames}  |  FPS: {dataset_fps}")
print(f"State column: '{STATE_COL}' {state_shape}  |  Action column: '{ACTION_COL}' {action_shape}")
print(f"{'='*60}")

# modality.json
modality_path = meta_dir / "modality.json"
if modality_path.exists():
    with open(modality_path) as f:
        modality = json.load(f)
    print(f"\nModality:")
    for key, val in modality.items():
        if isinstance(val, dict):
            dims = list(val.keys())
            print(f"  {key}: {dims}")
        else:
            print(f"  {key}: {val}")

# episodes.jsonl
episodes_path = meta_dir / "episodes.jsonl"
episodes_meta = []
if episodes_path.exists():
    with open(episodes_path) as f:
        for line in f:
            episodes_meta.append(json.loads(line.strip()))

# tasks.jsonl
tasks_path = meta_dir / "tasks.jsonl"
tasks_meta = []
if tasks_path.exists():
    with open(tasks_path) as f:
        for line in f:
            tasks_meta.append(json.loads(line.strip()))
    print(f"\nTasks:")
    for t in tasks_meta:
        print(f"  [{t.get('task_index', '?')}] {t.get('task', '')}")

# replay 모드 호환성 체크
if DATASET_TYPE == "eef" and REPLAY_MODE == "joint":
    print(f"\n⚠️  WARNING: EEF dataset + joint replay mode는 지원되지 않습니다 (IK 필요).")
    print(f"   REPLAY_MODE를 'eef'로 자동 변경합니다.")
    REPLAY_MODE = "eef"
elif DATASET_TYPE == "joint" and REPLAY_MODE == "eef":
    print(f"\n✓ Joint dataset + EEF replay mode: FK 변환을 사용합니다.")

print(f"\n최종 Replay 설정: dataset={DATASET_TYPE}, replay_mode={REPLAY_MODE}")

---
## Step 3 — 에피소드 브라우저 + 데이터 로드

에피소드 목록을 확인하고, 선택한 에피소드의 parquet 파일을 로드합니다.

In [ ]:
# ---------- 에피소드 목록 ----------
if episodes_meta:
    print(f"{'Ep':>4s}  {'Frames':>6s}  {'Task'}")
    print("-" * 60)
    for ep in episodes_meta:
        idx = ep["episode_index"]
        length = ep.get("length", "?")
        task_list = ep.get("tasks", [])
        task_str = task_list[0] if task_list else ""
        marker = " <<<" if idx == EPISODE_INDEX else ""
        print(f"{idx:4d}  {length:6}  {task_str}{marker}")
    print(f"\n선택된 에피소드: {EPISODE_INDEX}")
else:
    print("episodes.jsonl 없음 — parquet 파일에서 직접 로드")

# ---------- parquet 로드 ----------
# LeRobot v2: data/chunk-000/episode_NNNNNN.parquet
chunk_idx = EPISODE_INDEX // 1000
parquet_path = DATASET_DIR / "data" / f"chunk-{chunk_idx:03d}" / f"episode_{EPISODE_INDEX:06d}.parquet"
assert parquet_path.exists(), f"Parquet not found: {parquet_path}"

df_episode = pd.read_parquet(parquet_path)
n_frames = len(df_episode)
print(f"\nLoaded: {parquet_path.name} ({n_frames} frames)")
print(f"Columns: {list(df_episode.columns)}")

# ---------- state/action 추출 ----------
states = np.stack(df_episode[STATE_COL].values).astype(np.float32)
actions = np.stack(df_episode[ACTION_COL].values).astype(np.float32)
print(f"States shape: {states.shape}  |  Actions shape: {actions.shape}")

# ---------- 프레임 범위 적용 ----------
frame_start = FRAME_START if FRAME_START is not None else 0
frame_end = FRAME_END if FRAME_END is not None else n_frames
frame_start = max(0, min(frame_start, n_frames))
frame_end = max(frame_start, min(frame_end, n_frames))

states_sel = states[frame_start:frame_end]
actions_sel = actions[frame_start:frame_end]
print(f"\nReplay range: frame {frame_start} ~ {frame_end} ({frame_end - frame_start} frames)")
print(f"First state:  {np.round(states_sel[0], 4)}")
print(f"First action: {np.round(actions_sel[0], 4)}")

---
## Step 4 — 시각화 (State/Action Trajectory + 이미지)

선택한 에피소드의 state/action 궤적과 샘플 이미지를 시각화합니다.

In [ ]:
# ---------- State/Action 궤적 시각화 ----------
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

t = np.arange(frame_start, frame_end) / REPLAY_FPS  # 시간 (초)

if DATASET_TYPE == "eef":
    eef_labels = ["x", "y", "z", "roll", "pitch", "yaw", "gripper"]

    ax = axes[0]
    ax.set_title("State (EEF 7D)")
    for j in range(3):
        ax.plot(t, states_sel[:, j], label=f"pos_{eef_labels[j]}")
    ax.set_ylabel("Position [m]")
    ax.legend(loc="upper left", fontsize=8)
    ax_r = ax.twinx()
    for j in range(3, 6):
        ax_r.plot(t, states_sel[:, j], "--", alpha=0.6, label=f"rot_{eef_labels[j]}")
    ax_r.set_ylabel("Euler [rad]")
    ax_r.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    ax.set_title("Action (EEF 7D)")
    for j in range(3):
        ax.plot(t, actions_sel[:, j], label=f"pos_{eef_labels[j]}")
    ax.set_ylabel("Position [m]")
    ax.legend(loc="upper left", fontsize=8)
    ax_r = ax.twinx()
    for j in range(3, 6):
        ax_r.plot(t, actions_sel[:, j], "--", alpha=0.6, label=f"rot_{eef_labels[j]}")
    ax_r.set_ylabel("Euler [rad]")
    ax_r.legend(loc="upper right", fontsize=8)

else:
    ax = axes[0]
    ax.set_title("State — Right Arm (7) + Gripper")
    for j in range(7):
        ax.plot(t, states_sel[:, j], label=f"right_{j}")
    ax.set_ylabel("Joint angle [rad]")
    ax.legend(loc="upper left", fontsize=7, ncol=2)
    ax_r = ax.twinx()
    ax_r.plot(t, states_sel[:, 14], "k-", alpha=0.7, label="R_gripper")
    ax_r.set_ylabel("Gripper [0-1]")
    ax_r.set_ylim(-0.1, 1.1)
    ax_r.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    ax.set_title("Action — Right Arm (7) + Gripper")
    for j in range(7):
        ax.plot(t, actions_sel[:, j], label=f"right_{j}")
    ax.set_ylabel("Joint angle [rad]")
    ax.legend(loc="upper left", fontsize=7, ncol=2)
    ax_r = ax.twinx()
    ax_r.plot(t, actions_sel[:, 14], "k-", alpha=0.7, label="R_gripper")
    ax_r.set_ylabel("Gripper [0-1]")
    ax_r.set_ylim(-0.1, 1.1)
    ax_r.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time [s]")
fig.suptitle(f"Episode {EPISODE_INDEX} — {DATASET_TYPE.upper()} dataset ({n_frames} frames @ {REPLAY_FPS} fps)", fontsize=13)
plt.tight_layout()
plt.show()

# ---------- Gripper 비교 ----------
grip_idx = 6 if DATASET_TYPE == "eef" else 14

fig2, ax2 = plt.subplots(figsize=(14, 3))
ax2.plot(t, states_sel[:, grip_idx], label="state gripper", linewidth=1.5)
ax2.plot(t, actions_sel[:, grip_idx], "--", label="action gripper", linewidth=1.5)
ax2.set_ylabel("Gripper [0-1]")
ax2.set_xlabel("Time [s]")
ax2.set_title("Gripper — State vs Action")
ax2.legend()
ax2.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()


---
## Step 5 — FK 함수 정의 (EEF 모드용)

rby1a URDF kinematic chain을 사용한 순방향 기구학(FK) 함수입니다.
`lap/scripts/convert_rby1_joint_to_eef.py`에서 포팅되었습니다.

- Joint 데이터셋 → EEF 모드 replay 시 on-the-fly FK 변환에 사용
- EEF 데이터셋에서 `eef_7d_to_se3()` 변환에 사용

In [ ]:
from scipy.spatial.transform import Rotation as _Rotation

# =====================================================================
# Forward Kinematics: base -> torso(6) -> right_arm(7) -> tool -> ee_right
# =====================================================================
_FK_CHAIN = [
    (np.array([0.0, 0.0, 0.2805]),           np.array([1.0, 0.0, 0.0])),  # torso_0
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 1.0, 0.0])),  # torso_1
    (np.array([0.0, 0.0, 0.350]),            np.array([0.0, 1.0, 0.0])),  # torso_2
    (np.array([0.0, 0.0, 0.350]),            np.array([0.0, 1.0, 0.0])),  # torso_3
    (np.array([0.0, 0.0, 0.0]),              np.array([1.0, 0.0, 0.0])),  # torso_4
    (np.array([0.0, 0.0, 0.309426548461]),   np.array([0.0, 0.0, 1.0])),  # torso_5
    (np.array([0.0, -0.220, 0.080073451539]),np.array([0.0, 0.93969262, -0.34202014])),  # right_arm_0
    (np.array([0.0, 0.0, 0.0]),              np.array([1.0, 0.0, 0.0])),  # right_arm_1
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 0.0, 1.0])),  # right_arm_2
    (np.array([0.031, 0.0, -0.276]),         np.array([0.0, 1.0, 0.0])),  # right_arm_3
    (np.array([-0.031, 0.0, -0.256]),        np.array([0.0, 0.0, 1.0])),  # right_arm_4
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 1.0, 0.0])),  # right_arm_5
    (np.array([0.0, 0.0, 0.0]),              np.array([0.0, 0.0, 1.0])),  # right_arm_6
]
_FK_TOOL_OFFSET = np.array([0.0, 0.0, -0.1087])
_FK_FT_OFFSET = np.array([0.0, 0.0, -0.0461])


def _rodrigues(axis: np.ndarray, angle: float) -> np.ndarray:
    """Rodrigues' rotation formula -> 3x3 rotation matrix."""
    a = axis / np.linalg.norm(axis)
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


def _hom(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Build 4x4 homogeneous transform."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T


def fk_base_to_ee(joint_angles_13: np.ndarray) -> np.ndarray:
    """base -> ee_right 4x4 transform. Input: [torso(6), right_arm(7)]."""
    T = np.eye(4)
    for i, (xyz, axis) in enumerate(_FK_CHAIN):
        T = T @ _hom(np.eye(3), xyz) @ _hom(_rodrigues(axis, joint_angles_13[i]), np.zeros(3))
    T = T @ _hom(np.eye(3), _FK_TOOL_OFFSET) @ _hom(np.eye(3), _FK_FT_OFFSET)
    return T


def compute_eef_from_joints(right_arm_7: np.ndarray, gripper: float) -> np.ndarray:
    """Right arm 7D joints + gripper -> 7D EEF [x,y,z,roll,pitch,yaw,gripper]."""
    chain_q = np.concatenate([TORSO_FIXED_RAD, np.asarray(right_arm_7, dtype=np.float64)])
    T = fk_base_to_ee(chain_q)
    pos = T[:3, 3]
    euler = _Rotation.from_matrix(T[:3, :3]).as_euler("xyz", degrees=False)
    return np.concatenate([pos, euler, [gripper]]).astype(np.float32)


def joint_16d_to_eef_7d(joint_16d: np.ndarray) -> np.ndarray:
    """Convert 16D joint vector to 7D EEF. [right_arm(7), left_arm(7), r_grip, l_grip] -> [x,y,z,r,p,y,grip]."""
    return compute_eef_from_joints(joint_16d[0:7], joint_16d[14])


def eef_7d_to_se3(eef_7d: np.ndarray) -> np.ndarray:
    """7D EEF [x,y,z,roll,pitch,yaw,...] -> 4x4 SE3 matrix."""
    R = _Rotation.from_euler("xyz", eef_7d[3:6]).as_matrix()
    return _hom(R, eef_7d[:3])


# ---------- replay용 action 변환 ----------
def prepare_replay_actions(actions_raw: np.ndarray, dataset_type: str, replay_mode: str) -> np.ndarray:
    """Prepare actions for replay.

    Returns:
        For joint mode: (N, 16) array [right_arm(7), left_arm(7), r_grip, l_grip]
        For eef mode:   (N, 7)  array [x, y, z, roll, pitch, yaw, gripper]
    """
    if dataset_type == "joint" and replay_mode == "joint":
        return actions_raw  # 16D -> 16D direct
    elif dataset_type == "eef" and replay_mode == "eef":
        return actions_raw  # 7D -> 7D direct
    elif dataset_type == "joint" and replay_mode == "eef":
        # 16D -> 7D via FK
        N = actions_raw.shape[0]
        result = np.empty((N, 7), dtype=np.float32)
        for i in range(N):
            result[i] = joint_16d_to_eef_7d(actions_raw[i])
        return result
    else:
        raise ValueError(f"Unsupported: dataset={dataset_type}, replay={replay_mode}")


# ---------- 변환 테스트 ----------
replay_actions = prepare_replay_actions(actions_sel, DATASET_TYPE, REPLAY_MODE)
print(f"Replay actions prepared: {replay_actions.shape}")
print(f"  dataset_type={DATASET_TYPE}, replay_mode={REPLAY_MODE}")
print(f"  first: {np.round(replay_actions[0], 4)}")
print(f"  last:  {np.round(replay_actions[-1], 4)}")

if REPLAY_MODE == "eef":
    pos_range = replay_actions[:, :3].max(axis=0) - replay_actions[:, :3].min(axis=0)
    print(f"  EEF position range: {np.round(pos_range, 4)} m")
    if np.any(pos_range > 1.0):
        print("  WARNING: EEF position range > 1m — check FK/data!")

---
## Step 6 — 로봇 연결 + 초기 자세 이동

rby1에 연결하고, 안전한 경유점 경로를 통해 데이터 수집 시의 초기 자세로 이동합니다.

> **안전 경로**: 팔꿈치가 펴져 있으면 midpoint1 → midpoint2 → initial, 이미 구부러져 있으면 midpoint2 → initial

In [ ]:
import rby1_sdk as rby

# ---------- 로봇 연결 ----------
robot = rby.create_robot(_connect_addr, ROBOT_MODEL) if hasattr(rby, "create_robot") else rby.create_robot_a(_connect_addr)
robot.connect()
assert robot.is_connected(), f"Failed to connect robot at {_connect_addr}"

robot.power_on(".*")
robot.servo_on(".*")
if robot.get_control_manager_state().state in [
    rby.ControlManagerState.State.MajorFault,
    rby.ControlManagerState.State.MinorFault,
]:
    robot.reset_fault_control_manager()
robot.enable_control_manager()

try:
    robot.cancel_control()
except Exception:
    pass
robot.wait_for_control_ready(1000)

print(f"[robot] Connected [{ROBOT_MODE.upper()}]: {_connect_addr}")
q0 = np.asarray(robot.get_state().position, dtype=np.float64)
print(f"[robot] current torso: {np.round(np.degrees(q0[TORSO_SLICE]), 1)} deg")
print(f"[robot] current right: {np.round(np.degrees(q0[RIGHT_SLICE]), 1)} deg")
print(f"[robot] current left:  {np.round(np.degrees(q0[LEFT_SLICE]), 1)} deg")


# ---------- safe path 함수 ----------
def _build_body_cmd(torso_q, right_q, left_q, dt, hold_time):
    body_builder = rby.BodyComponentBasedCommandBuilder()
    if torso_q is not None:
        body_builder = body_builder.set_torso_command(
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
            .set_position(np.asarray(torso_q, dtype=np.float64))
            .set_minimum_time(float(dt))
        )
    body_builder = (
        body_builder
        .set_right_arm_command(
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
            .set_position(np.asarray(right_q, dtype=np.float64))
            .set_minimum_time(float(dt))
        )
        .set_left_arm_command(
            rby.JointPositionCommandBuilder()
            .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(hold_time))
            .set_position(np.asarray(left_q, dtype=np.float64))
            .set_minimum_time(float(dt))
        )
    )
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_body_command(body_builder)
    )


def move_waypoint(robot, stream, name, torso_target, right_target, left_target, duration):
    q_now = np.asarray(robot.get_state().position, dtype=np.float64)
    start_torso = q_now[TORSO_SLICE].copy()
    start_right = q_now[RIGHT_SLICE].copy()
    start_left  = q_now[LEFT_SLICE].copy()
    goal_torso = torso_target if torso_target is not None else start_torso
    steps = max(2, int(np.ceil(duration / INIT_DT)))
    torso_path = np.linspace(start_torso, goal_torso, num=steps)
    right_path  = np.linspace(start_right, right_target, num=steps)
    left_path   = np.linspace(start_left,  left_target,  num=steps)
    for i in range(steps):
        cmd = _build_body_cmd(torso_path[i], right_path[i], left_path[i], INIT_DT, INIT_HOLD_TIME)
        try:
            stream.send_command(cmd)
        except RuntimeError as exc:
            if "expired" in str(exc).lower():
                robot.wait_for_control_ready(1000)
                stream = robot.create_command_stream(priority=COMMAND_PRIORITY)
                stream.send_command(cmd)
            else:
                raise
        time.sleep(INIT_DT)
    q_after = np.asarray(robot.get_state().position, dtype=np.float64)
    print(f"  -> {name} done | right[0]={np.degrees(q_after[8]):+.1f} deg")
    return stream


# ---------- 초기 자세 이동 ----------
right_elbow = float(q0[RIGHT_SLICE][3])
left_elbow  = float(q0[LEFT_SLICE][3])
elbow_bent  = (
    abs(right_elbow) > np.deg2rad(ELBOW_THRESHOLD_DEG)
    or abs(left_elbow) > np.deg2rad(ELBOW_THRESHOLD_DEG)
)
print(f"\n[init] elbow check: right={np.degrees(right_elbow):+.1f}, left={np.degrees(left_elbow):+.1f} deg")
print(f"[init] elbow {'bent' if elbow_bent else 'straight'} -> ", end="")

if elbow_bent:
    waypoints = [
        ("midpoint2", None,        MID2_RIGHT, MID2_LEFT, 5.0),
        ("initial",   TORSO_INIT,  INIT_RIGHT, INIT_LEFT, 3.0),
    ]
    print("midpoint2 -> initial")
else:
    waypoints = [
        ("midpoint1", None,        MID1_RIGHT, MID1_LEFT, 5.0),
        ("midpoint2", None,        MID2_RIGHT, MID2_LEFT, 5.0),
        ("initial",   TORSO_INIT,  INIT_RIGHT, INIT_LEFT, 3.0),
    ]
    print("midpoint1 -> midpoint2 -> initial")

# tool flange 12V — 실로봇에서만 활성화 (sim 모드 skip)
if USE_REMOTE_GRIPPER and ROBOT_MODE == "real":
    for arm in ("right", "left"):
        try:
            robot.set_tool_flange_output_voltage(arm, 12)
        except Exception:
            pass
    time.sleep(0.5)
    print("[flange] 12V activated")
elif ROBOT_MODE == "sim":
    print("[flange] sim 모드 — tool flange 전압 설정 skip")

stream = robot.create_command_stream(priority=COMMAND_PRIORITY)
for name, torso_t, right_t, left_t, dur in waypoints:
    print(f"[init] moving -> {name} ({dur:.1f}s)")
    stream = move_waypoint(robot, stream, name, torso_t, right_t, left_t, dur)

time.sleep(0.2)
q_end = np.asarray(robot.get_state().position, dtype=np.float64)
err_r = float(np.linalg.norm(q_end[RIGHT_SLICE] - INIT_RIGHT))
err_l = float(np.linalg.norm(q_end[LEFT_SLICE]  - INIT_LEFT))
print(f"\n[init] done | error right={err_r:.6f}, left={err_l:.6f}")

try:
    stream.cancel()
except Exception:
    pass
try:
    robot.cancel_control()
except Exception:
    pass

INIT_DONE = True
print("[init] complete")


---
## Step 7 — Gripper 초기화

Remote gripper (UDP) 연결 → homing → start → 완전 열기

In [ ]:
# ---------- Gripper 초기화 ----------
gripper_obj = None

# sim 모드에서는 물리 그리퍼 없음 → 전체 skip
if ROBOT_MODE == "sim":
    GRIPPER_READY = False
    print("[gripper] sim 모드 — gripper 초기화 skip")
elif USE_REMOTE_GRIPPER:
    from remote_gripper import Gripper as RemoteGripper

    gripper_obj = RemoteGripper()
    print(f"[gripper] host={gripper_obj.host}  port={gripper_obj.port}")

    ok = gripper_obj.initialize(verbose=True)
    if not ok:
        raise RuntimeError("[gripper] 서버 응답 없음")
    print("[gripper] ping OK")

    print("[gripper] homing...")
    ok = gripper_obj.homing()
    if not ok:
        raise RuntimeError("[gripper] homing 실패")
    print(f"[gripper] homing done | min_q={gripper_obj.min_q}, max_q={gripper_obj.max_q}")

    gripper_obj.start()
    print("[gripper] start()")

    gripper_obj.set_normalized_target(np.array([1.0, 1.0]), wait_for_reply=True)
    time.sleep(0.5)
    print("[gripper] fully open")

    GRIPPER_READY = True
else:
    GRIPPER_READY = False
    print("[gripper] skipped (USE_REMOTE_GRIPPER=False)")


---
## Step 8 — Replay 실행

선택한 에피소드의 action 시퀀스를 로봇에 순차 전송합니다.

### Joint 모드
- 16D action에서 right_arm(7) + gripper 추출
- `JointPositionCommand`로 right arm 제어 (torso/left arm hold)

### EEF 모드
- 7D action에서 [x,y,z,roll,pitch,yaw] → `SE3` → `CartesianCommand`
- SDK 내부 IK로 right arm 제어 (torso/left arm hold)

In [ ]:
# ---------- Replay 실행 ----------
if not globals().get("INIT_DONE", False):
    raise RuntimeError("먼저 Step 6 (초기 자세 이동)을 실행하세요.")

n_replay = len(replay_actions)
replay_dt = 1.0 / (REPLAY_FPS * REPLAY_SPEED)

print(f"{'='*60}")
print(f"Replay Episode {EPISODE_INDEX}  [{ROBOT_MODE.upper()}]")
print(f"  Mode: {REPLAY_MODE}  |  Dataset: {DATASET_TYPE}")
print(f"  Frames: {n_replay}  |  FPS: {REPLAY_FPS}  |  Speed: {REPLAY_SPEED}x")
print(f"  DT: {replay_dt:.4f}s  |  Duration: {n_replay * replay_dt:.1f}s")
print(f"  Gripper: {'enabled' if (gripper_obj is not None and ROBOT_MODE == 'real') else 'disabled (sim or off)'}")
print(f"{'='*60}")

# ---------- Robot command builders ----------
def build_joint_cmd(torso_hold, right_target, left_hold):
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_body_command(
            rby.BodyComponentBasedCommandBuilder()
            .set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .set_position(torso_hold.astype(np.float64))
                .set_minimum_time(MINIMUM_TIME)
            )
            .set_right_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .set_position(right_target.astype(np.float64))
                .set_minimum_time(MINIMUM_TIME)
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .set_position(left_hold.astype(np.float64))
                .set_minimum_time(MINIMUM_TIME)
            )
        )
    )


def build_cartesian_cmd(torso_hold, T_target, left_hold):
    return rby.RobotCommandBuilder().set_command(
        rby.ComponentBasedCommandBuilder().set_body_command(
            rby.BodyComponentBasedCommandBuilder()
            .set_torso_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .set_position(torso_hold.astype(np.float64))
                .set_minimum_time(0.1)
            )
            .set_right_arm_command(
                rby.CartesianCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .add_target("base", "ee_right", T_target.astype(np.float64),
                            EEF_TRANSLATION_WEIGHT, EEF_ROTATION_WEIGHT, EEF_VELOCITY_SCALING)
            )
            .set_left_arm_command(
                rby.JointPositionCommandBuilder()
                .set_command_header(rby.CommandHeaderBuilder().set_control_hold_time(HOLD_TIME))
                .set_position(left_hold.astype(np.float64))
                .set_minimum_time(0.1)
            )
        )
    )


# ---------- 로봇 상태 준비 ----------
robot.wait_for_control_ready(1000)
stream = robot.create_command_stream(priority=COMMAND_PRIORITY)

q_now = np.asarray(robot.get_state().position, dtype=np.float64)
torso_hold     = q_now[TORSO_SLICE].copy()
left_hold      = q_now[LEFT_SLICE].copy()
ep_start_right = q_now[RIGHT_SLICE].copy()

print(f"\n[replay] torso hold:  {np.round(np.degrees(torso_hold), 1)} deg")
print(f"[replay] left hold:   {np.round(np.degrees(left_hold), 1)} deg")
print(f"[replay] right start: {np.round(np.degrees(ep_start_right), 1)} deg")

if REPLAY_MODE == "eef":
    init_eef = compute_eef_from_joints(ep_start_right, 1.0)
    print(f"[replay] initial EEF: pos={np.round(init_eef[:3], 4)} euler={np.round(init_eef[3:6], 4)}")

# ---------- 로깅 버퍼 ----------
_log_cmd        = []
_log_actual     = []
_log_gripper_cmd = []
_log_time       = []

# ---------- Replay loop ----------
send_errors = 0
t_start = time.time()

print(f"\n[replay] starting...")
for i in range(n_replay):
    t0 = time.time()
    action = replay_actions[i]

    if REPLAY_MODE == "joint":
        right_target = action[0:7]
        gripper_val  = float(action[14])
        cmd = build_joint_cmd(torso_hold, right_target, left_hold)
        _log_cmd.append(right_target.copy())
    else:
        T_target    = eef_7d_to_se3(action)
        gripper_val = float(action[6])
        cmd = build_cartesian_cmd(torso_hold, T_target, left_hold)
        _log_cmd.append(action[:6].copy())

    # 커맨드 전송
    try:
        stream.send_command(cmd)
    except RuntimeError as exc:
        send_errors += 1
        if "expired" in str(exc).lower():
            robot.wait_for_control_ready(1000)
            stream = robot.create_command_stream(priority=COMMAND_PRIORITY)
            stream.send_command(cmd)
        else:
            raise

    # 그리퍼 — 실로봇에서만 전송 (sim 모드 skip)
    if gripper_obj is not None and ROBOT_MODE == "real":
        try:
            gripper_obj.set_normalized_target(
                np.array([gripper_val, 1.0]), wait_for_reply=False
            )
        except Exception:
            pass
    _log_gripper_cmd.append(gripper_val)

    # 실제 상태 로깅
    q_actual = np.asarray(robot.get_state().position, dtype=np.float64)
    if REPLAY_MODE == "joint":
        _log_actual.append(q_actual[RIGHT_SLICE].copy())
    else:
        actual_eef = compute_eef_from_joints(q_actual[RIGHT_SLICE], gripper_val)
        _log_actual.append(actual_eef[:6].copy())
    _log_time.append(time.time() - t_start)

    # 진행상황 출력
    if i == 0 or (i + 1) % 20 == 0 or (i + 1) == n_replay:
        if REPLAY_MODE == "joint":
            err = float(np.linalg.norm(action[0:7] - q_actual[RIGHT_SLICE]))
            print(f"  step {i+1:4d}/{n_replay} | joint_err={err:.4f} | gripper={gripper_val:.3f}")
        else:
            actual_eef_full = compute_eef_from_joints(q_actual[RIGHT_SLICE], gripper_val)
            pos_err = float(np.linalg.norm(action[:3] - actual_eef_full[:3]))
            print(f"  step {i+1:4d}/{n_replay} | pos_err={pos_err:.5f}m | gripper={gripper_val:.3f}")

    elapsed = time.time() - t0
    time.sleep(max(0, replay_dt - elapsed))

total_time = time.time() - t_start

# ---------- 결과 요약 ----------
print(f"\n{'='*60}")
print(f"Replay Complete  [{ROBOT_MODE.upper()}]")
print(f"  Frames: {n_replay}  |  Duration: {total_time:.1f}s  |  Avg FPS: {n_replay/total_time:.1f}")
print(f"  Send errors: {send_errors}")

q_final = np.asarray(robot.get_state().position, dtype=np.float64)
moved = float(np.linalg.norm(q_final[RIGHT_SLICE] - ep_start_right))
print(f"  Total joint movement: {moved:.4f} rad")

if REPLAY_MODE == "eef":
    final_eef = compute_eef_from_joints(q_final[RIGHT_SLICE], 1.0)
    eef_moved = float(np.linalg.norm(final_eef[:3] - init_eef[:3]))
    print(f"  Total EEF movement: {eef_moved:.4f} m")
    print(f"  Final EEF: pos={np.round(final_eef[:3], 4)} euler={np.round(final_eef[3:6], 4)}")

print(f"{'='*60}")


---
## Step 9 — Replay 결과 분석

명령(commanded) vs 실제(actual) 궤적을 비교하고 tracking error를 분석합니다.

In [ ]:
# ---------- Replay 결과 시각화 ----------
log_cmd = np.array(_log_cmd)
log_actual = np.array(_log_actual)
log_gripper = np.array(_log_gripper_cmd)
log_time = np.array(_log_time)

# tracking error per step
if REPLAY_MODE == "joint":
    errors = np.linalg.norm(log_cmd - log_actual, axis=1)
    error_label = "Joint tracking error [rad]"
else:
    pos_errors = np.linalg.norm(log_cmd[:, :3] - log_actual[:, :3], axis=1)
    rot_errors = np.linalg.norm(log_cmd[:, 3:6] - log_actual[:, 3:6], axis=1)
    errors = pos_errors
    error_label = "EEF position tracking error [m]"

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# (1) Commanded vs Actual trajectory
ax = axes[0]
if REPLAY_MODE == "eef":
    for j, label in enumerate(["x", "y", "z"]):
        ax.plot(log_time, log_cmd[:, j], "-", label=f"cmd_{label}", alpha=0.8)
        ax.plot(log_time, log_actual[:, j], "--", label=f"actual_{label}", alpha=0.8)
    ax.set_ylabel("Position [m]")
    ax.set_title("Commanded vs Actual — EEF Position")
else:
    for j in range(min(7, log_cmd.shape[1])):
        ax.plot(log_time, log_cmd[:, j], "-", alpha=0.6, label=f"cmd_{j}")
        ax.plot(log_time, log_actual[:, j], "--", alpha=0.6, label=f"actual_{j}")
    ax.set_ylabel("Joint [rad]")
    ax.set_title("Commanded vs Actual — Right Arm Joints")
ax.legend(loc="upper left", fontsize=7, ncol=3)

# (2) Tracking error
ax = axes[1]
ax.plot(log_time, errors, "r-", linewidth=1.0)
ax.axhline(np.mean(errors), color="r", linestyle="--", alpha=0.5, label=f"mean={np.mean(errors):.5f}")
ax.set_ylabel(error_label)
ax.set_title("Tracking Error")
ax.legend()

if REPLAY_MODE == "eef":
    ax_rot = ax.twinx()
    ax_rot.plot(log_time, rot_errors, "b-", linewidth=0.8, alpha=0.6, label=f"rot_err (mean={np.mean(rot_errors):.4f})")
    ax_rot.set_ylabel("Rotation error [rad]")
    ax_rot.legend(loc="upper right")

# (3) Gripper
ax = axes[2]
ax.plot(log_time, log_gripper, "g-", linewidth=1.5, label="gripper cmd")
ax.set_ylabel("Gripper [0-1]")
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel("Time [s]")
ax.set_title("Gripper Command")
ax.legend()

fig.suptitle(
    f"Replay Analysis — Ep {EPISODE_INDEX} | {REPLAY_MODE.upper()} mode | "
    f"{n_replay} frames @ {REPLAY_SPEED}x speed",
    fontsize=13,
)
plt.tight_layout()
plt.show()

# ---------- 통계 요약 ----------
print(f"\nTracking Error Summary ({error_label}):")
print(f"  Mean:   {np.mean(errors):.6f}")
print(f"  Median: {np.median(errors):.6f}")
print(f"  Max:    {np.max(errors):.6f}")
print(f"  Std:    {np.std(errors):.6f}")
if REPLAY_MODE == "eef":
    print(f"\nRotation Error Summary [rad]:")
    print(f"  Mean:   {np.mean(rot_errors):.6f}")
    print(f"  Max:    {np.max(rot_errors):.6f}")

# timing jitter
dt_actual = np.diff(log_time)
print(f"\nTiming:")
print(f"  Target DT: {replay_dt:.4f}s")
print(f"  Actual DT: mean={np.mean(dt_actual):.4f}s, std={np.std(dt_actual):.4f}s")
print(f"  Max jitter: {np.max(np.abs(dt_actual - replay_dt)):.4f}s")

---
## Step 10 — Cleanup

로봇 제어 해제 및 연결 종료

In [ ]:
# ---------- Cleanup ----------
try:
    robot.cancel_control()
    print("[cleanup] cancel_control done")
except Exception:
    pass

if gripper_obj is not None:
    try:
        gripper_obj.stop()
        print("[cleanup] gripper stopped")
    except Exception:
        pass

if hasattr(robot, "disconnect"):
    robot.disconnect()
    print("[cleanup] robot disconnected")

print("[cleanup] complete")